# R2.2 + R2.4 Experiments — ICMLDE 2026 LULC Paper

**Fully self-contained — no project repo needed.**

Dataset path expected:
```
/kaggle/input/uc-merced-land-use-dataset/UCMerced_LandUse/Images/<classname>/*.tif
```

This notebook:
1. Loads all 2100 UC Merced images
2. Extracts **Branch A** handcrafted features (GLCM, LBP, Haralick, colour moments, Hu moments) — exactly as described in the paper
3. Extracts **Branch B** deep features from frozen EfficientNet-B0 (1280-d), ResNet-50 (2048-d), MobileNetV3-Large (960-d)
4. Concatenates into 4370-d hybrid vector
5. **R2.2**: Ablation — trains RF + XGBoost on full / deep-only / handcrafted-only
6. **R2.4**: Multi-seed SHAP — runs SHAP group attribution across 5 seeds, reports mean ± std and CV

Outputs two CSVs to `/kaggle/working/`:
- `r22_ablation_results.csv`
- `r24_shap_consistency.csv`

In [ ]:
!pip install -q shap scikit-image

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
import cv2
import torch
import torchvision.models as tvm
import torchvision.transforms as T
from PIL import Image
from tqdm import tqdm
from scipy.stats import skew
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
import shap
import warnings
warnings.filterwarnings('ignore')

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
IMG_DIR  = "/kaggle/input/datasets/srinivasarukonda/uc-merced-land-use-dataset/UCMerced_LandUse/Images"
OUTDIR   = "/kaggle/working"
IMG_SIZE = 224          # backbone input size
SEEDS    = [42, 0, 7, 123, 999]
BASE_SEED = 42

print(f"Device: {DEVICE}")
print(f"Image dir exists: {os.path.isdir(IMG_DIR)}")

## 1. Load images

In [ ]:
class_names = sorted([
    d for d in os.listdir(IMG_DIR)
    if os.path.isdir(os.path.join(IMG_DIR, d))
])
print(f"{len(class_names)} classes: {class_names}")

image_paths, labels = [], []
for label_id, cname in enumerate(class_names):
    files = sorted(glob.glob(os.path.join(IMG_DIR, cname, "*.tif")))
    image_paths.extend(files)
    labels.extend([label_id] * len(files))

labels = np.array(labels)
print(f"Total images: {len(image_paths)}, labels shape: {labels.shape}")

## 2. Branch A — Handcrafted features (82-d)
GLCM texture, LBP, Haralick, colour moments (RGB+HSV), Hu shape moments — exactly as in paper.

In [ ]:
def extract_handcrafted(img_bgr):
    """img_bgr: uint8 numpy array (H,W,3) in BGR (from cv2.imread).
    Returns 1-D float32 vector of length 82.
    """
    feats = []

    # -- GLCM texture (contrast, dissimilarity, homogeneity, energy, correlation, ASM)
    # 4 orientations: 0, 45, 90, 135 degrees
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    glcm = graycomatrix(
        gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256, symmetric=True, normed=True
    )
    for prop in ['contrast','dissimilarity','homogeneity','energy','correlation','ASM']:
        vals = graycoprops(glcm, prop)   # shape (1, 4)
        feats.extend(vals.flatten().tolist())
    # 6 props x 4 angles = 24 features

    # -- LBP histogram (uniform, radius=1, P=8)
    lbp = local_binary_pattern(gray, P=8, R=1, method='uniform')
    hist, _ = np.histogram(lbp, bins=10, range=(0, 10), density=True)
    feats.extend(hist.tolist())
    # 10 features

    # -- Haralick (inertia, correlation, energy, homogeneity) — mean over 4 angles
    for prop in ['contrast','correlation','energy','homogeneity']:
        feats.append(float(graycoprops(glcm, prop).mean()))
    # 4 features

    # -- Colour moments: mean, std, skewness for each of R,G,B,H,S,V channels
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).astype(np.float32)
    img_hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    for ch_img in [img_rgb, img_hsv]:
        for c in range(3):
            ch = ch_img[:, :, c].ravel()
            feats.append(float(ch.mean()))
            feats.append(float(ch.std()))
            feats.append(float(skew(ch)))
    # 2 colour spaces x 3 channels x 3 moments = 18 features

    # -- Hu invariant shape moments (7)
    moments = cv2.moments(gray)
    hu = cv2.HuMoments(moments).flatten()
    # log-transform to bring into reasonable range
    hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)
    feats.extend(hu.tolist())
    # 7 features

    # Total: 24 + 10 + 4 + 18 + 7 = 63 ... pad to 82 with additional GLCM at distance=2
    glcm2 = graycomatrix(
        gray, distances=[2], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=256, symmetric=True, normed=True
    )
    for prop in ['contrast','dissimilarity','homogeneity','energy','ASM']:
        vals2 = graycoprops(glcm2, prop)
        feats.extend(vals2.flatten().tolist())
    # 5 props x 4 angles = 20 features -> total = 63 + 20 = 83, drop last 1

    arr = np.array(feats, dtype=np.float32)
    arr = arr[:82]   # ensure exactly 82
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return arr


print("Extracting handcrafted features for all 2100 images...")
X_hc = []
for path in tqdm(image_paths):
    img = cv2.imread(path)
    if img is None:                         # tif fallback via PIL
        pil = Image.open(path).convert("RGB")
        img = cv2.cvtColor(np.array(pil), cv2.COLOR_RGB2BGR)
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    X_hc.append(extract_handcrafted(img))

X_hc = np.vstack(X_hc)
print(f"Handcrafted features shape: {X_hc.shape}")   # should be (2100, 82)

## 3. Branch B — Deep features (4288-d)
EfficientNet-B0 (1280) + ResNet-50 (2048) + MobileNetV3-Large (960) — frozen, ImageNet pretrained.

In [ ]:
transform = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

def load_backbone(name):
    if name == "efficientnet":
        m = tvm.efficientnet_b0(weights=tvm.EfficientNet_B0_Weights.IMAGENET1K_V1)
        m.classifier = torch.nn.Identity()
    elif name == "resnet50":
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = torch.nn.Identity()
    elif name == "mobilenet":
        m = tvm.mobilenet_v3_large(weights=tvm.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
        m.classifier = torch.nn.Identity()
    m.eval()
    for p in m.parameters():
        p.requires_grad_(False)
    return m.to(DEVICE)

def extract_deep(backbone, paths, batch_size=32):
    all_feats = []
    for i in tqdm(range(0, len(paths), batch_size)):
        batch_paths = paths[i:i+batch_size]
        imgs = []
        for p in batch_paths:
            try:
                pil = Image.open(p).convert("RGB")
            except Exception:
                pil = Image.fromarray(cv2.cvtColor(
                    cv2.imread(p), cv2.COLOR_BGR2RGB))
            imgs.append(transform(pil))
        batch = torch.stack(imgs).to(DEVICE)
        with torch.no_grad():
            feats = backbone(batch).cpu().numpy()
        # MobileNetV3 classifier=Identity returns shape (B, 960, 1, 1) — squeeze
        if feats.ndim > 2:
            feats = feats.reshape(feats.shape[0], -1)
        all_feats.append(feats)
    return np.vstack(all_feats)

print("Loading EfficientNet-B0...")
eff = load_backbone("efficientnet")
X_eff = extract_deep(eff, image_paths)
print(f"EfficientNet features: {X_eff.shape}")   # (2100, 1280)
del eff; torch.cuda.empty_cache()

print("Loading ResNet-50...")
res = load_backbone("resnet50")
X_res = extract_deep(res, image_paths)
print(f"ResNet-50 features: {X_res.shape}")       # (2100, 2048)
del res; torch.cuda.empty_cache()

print("Loading MobileNetV3-Large...")
mob = load_backbone("mobilenet")
X_mob = extract_deep(mob, image_paths)
print(f"MobileNetV3 features: {X_mob.shape}")     # (2100, 960)
del mob; torch.cuda.empty_cache()

In [ ]:
# Concatenate full hybrid vector
X_deep   = np.concatenate([X_eff, X_res, X_mob], axis=1)   # (2100, 4288)
X_hybrid = np.concatenate([X_hc, X_deep], axis=1)           # (2100, 4370)

HC_DIM   = X_hc.shape[1]     # 82
DEEP_DIM = X_deep.shape[1]   # 4288

print(f"hc_dim={HC_DIM}, deep_dim={DEEP_DIM}, hybrid={X_hybrid.shape[1]}")
assert X_hybrid.shape == (2100, 4370), f"Unexpected shape: {X_hybrid.shape}"

# Save features so we don't re-extract for each seed
np.save(f"{OUTDIR}/X_hybrid.npy", X_hybrid)
np.save(f"{OUTDIR}/y.npy", labels)
print("Features saved.")

---
## R2.2 — Ablation Study
Train RF + XGBoost on three configurations. Same hyperparameters as the paper.

In [ ]:
try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not available")

N_CLASSES = len(class_names)

def make_rf(seed=42):
    return RandomForestClassifier(
        n_estimators=500, max_features="sqrt",
        min_samples_leaf=2, class_weight="balanced",
        random_state=seed, n_jobs=-1
    )

def make_xgb(seed=42):
    params = dict(
        n_estimators=500, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        objective="multi:softprob", num_class=N_CLASSES,
        eval_metric="mlogloss", random_state=seed,
        verbosity=0,
    )
    if torch.cuda.is_available():
        params["device"] = "cuda"
    return XGBClassifier(**params)

# Fixed train/test split (same seed as paper)
X_tr, X_te, y_tr, y_te = train_test_split(
    X_hybrid, labels, test_size=0.20,
    random_state=BASE_SEED, stratify=labels
)

# Scale
scaler = StandardScaler().fit(X_tr)
X_tr_s = scaler.transform(X_tr)
X_te_s = scaler.transform(X_te)

ablation_configs = {
    f"Full Hybrid ({HC_DIM+DEEP_DIM}-d)"  : (X_tr_s,                   X_te_s),
    f"Deep-Only ({DEEP_DIM}-d)"            : (X_tr_s[:, HC_DIM:],      X_te_s[:, HC_DIM:]),
    f"Handcrafted-Only ({HC_DIM}-d)"       : (X_tr_s[:, :HC_DIM],      X_te_s[:, :HC_DIM]),
}

rows = []
for cfg_name, (Xtr, Xte) in ablation_configs.items():
    print(f"\n--- {cfg_name} ---")

    rf = make_rf(BASE_SEED)
    rf.fit(Xtr, y_tr)
    rf_pred = rf.predict(Xte)
    rf_acc  = accuracy_score(y_te, rf_pred)
    rf_f1   = f1_score(y_te, rf_pred, average="macro")
    print(f"  RF   acc={rf_acc:.4f}  macro-F1={rf_f1:.4f}")
    rows.append({"configuration": cfg_name, "classifier": "RandomForest",
                 "accuracy": rf_acc, "macro_f1": rf_f1})

    if HAS_XGB:
        xgb = make_xgb(BASE_SEED)
        xgb.fit(Xtr, y_tr)
        xgb_pred = xgb.predict(Xte)
        xgb_acc  = accuracy_score(y_te, xgb_pred)
        xgb_f1   = f1_score(y_te, xgb_pred, average="macro")
        print(f"  XGB  acc={xgb_acc:.4f}  macro-F1={xgb_f1:.4f}")
        rows.append({"configuration": cfg_name, "classifier": "XGBoost",
                     "accuracy": xgb_acc, "macro_f1": xgb_f1})

df_ablation = pd.DataFrame(rows)
df_ablation.to_csv(f"{OUTDIR}/r22_ablation_results.csv", index=False)
print("\n=== R2.2 ABLATION RESULTS ===")
print(df_ablation.to_string(index=False))

---
## R2.4 — SHAP Statistical Consistency (5 seeds)
Re-train RF on each seed, compute SHAP group attribution, report mean ± std and CV.

In [ ]:
def mean_abs_shap(sv):
    """Handle both list-of-arrays (multi-class) and single 2-D array."""
    if isinstance(sv, list):
        return np.mean(np.abs(np.stack(sv, axis=0)), axis=(0, 1))
    return np.mean(np.abs(sv), axis=0)

seed_rows = []

for seed in SEEDS:
    print(f"\n--- Seed {seed} ---")

    # Different train/test split per seed
    Xtr_s2, Xte_s2, ytr_s2, yte_s2 = train_test_split(
        X_hybrid, labels, test_size=0.20,
        random_state=seed, stratify=labels
    )
    sc2 = StandardScaler().fit(Xtr_s2)
    Xtr_s2 = sc2.transform(Xtr_s2)
    Xte_s2 = sc2.transform(Xte_s2)

    # --- Random Forest ---
    rf2 = make_rf(seed)
    rf2.fit(Xtr_s2, ytr_s2)
    explainer = shap.TreeExplainer(rf2)
    # Use a subsample of test set for speed (SHAP on RF is slow for 21 classes)
    rng = np.random.RandomState(seed)
    idx = rng.choice(len(Xte_s2), min(200, len(Xte_s2)), replace=False)
    sv = explainer.shap_values(Xte_s2[idx])
    ma = mean_abs_shap(sv)
    hc_pct   = 100 * ma[:HC_DIM].sum()   / ma.sum()
    deep_pct = 100 * ma[HC_DIM:].sum()   / ma.sum()
    print(f"  RF   Deep={deep_pct:.2f}%  Handcrafted={hc_pct:.2f}%")
    seed_rows.append({"seed": seed, "classifier": "RandomForest",
                      "deep_pct": deep_pct, "handcrafted_pct": hc_pct})

    # --- XGBoost ---
    if HAS_XGB:
        xgb2 = make_xgb(seed)
        xgb2.fit(Xtr_s2, ytr_s2)
        explainer_x = shap.TreeExplainer(xgb2)
        sv_x = explainer_x.shap_values(Xte_s2[idx])
        ma_x = mean_abs_shap(sv_x)
        hc_pct_x   = 100 * ma_x[:HC_DIM].sum() / ma_x.sum()
        deep_pct_x = 100 * ma_x[HC_DIM:].sum() / ma_x.sum()
        print(f"  XGB  Deep={deep_pct_x:.2f}%  Handcrafted={hc_pct_x:.2f}%")
        seed_rows.append({"seed": seed, "classifier": "XGBoost",
                          "deep_pct": deep_pct_x, "handcrafted_pct": hc_pct_x})

df_seeds = pd.DataFrame(seed_rows)
df_seeds.to_csv(f"{OUTDIR}/r24_shap_per_seed.csv", index=False)
print("\nPer-seed results saved.")

In [ ]:
# Compute mean, std, CV per classifier per group
consistency_rows = []
for clf_name, grp in df_seeds.groupby("classifier"):
    for col, label in [("deep_pct", "Deep"), ("handcrafted_pct", "Handcrafted")]:
        mean_v = grp[col].mean()
        std_v  = grp[col].std()
        cv     = (std_v / mean_v) * 100
        consistency_rows.append({
            "classifier": clf_name, "group": label,
            "mean_%": round(mean_v, 2),
            "std_%":  round(std_v,  2),
            "CV_%":   round(cv,     2),
        })

df_cons = pd.DataFrame(consistency_rows)
df_cons.to_csv(f"{OUTDIR}/r24_shap_consistency.csv", index=False)

print("=== R2.4 SHAP CONSISTENCY ===")
print(df_cons.to_string(index=False))

print("\n=== SUMMARY FOR PAPER ===")
print("\nR2.2 Ablation:")
print(df_ablation.to_string(index=False))
print("\nR2.4 SHAP Consistency:")
print(df_cons.to_string(index=False))
print(f"\nFiles saved to {OUTDIR}/")
print("  r22_ablation_results.csv")
print("  r24_shap_consistency.csv")
print("  r24_shap_per_seed.csv")